## 🩺 Medical Cost Prediction

Given *patient data*, let's try to predict the **charges** a given patient will incur.

We will use a variety of linear regression models to make our predictions.

Data source: https://www.kaggle.com/datasets/mirichoi0218/insurance

### Importing Libraries

In [28]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV

In [2]:
data = pd.read_csv('insurance.csv')
data

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


### Preprocessing

In [7]:
print("Total missing values:", data.isna().sum().sum())

Total missing values: 0


In [10]:
print("Total non-numeric columns:", len(data.select_dtypes('str').columns))

Total non-numeric columns: 3


In [12]:
data['children'] = data['children'].astype(str)

In [13]:
{column: list(data[column].unique()) for column in data.select_dtypes('str').columns}

{'sex': ['female', 'male'],
 'children': ['0', '1', '3', '2', '5', '4'],
 'smoker': ['yes', 'no'],
 'region': ['southwest', 'southeast', 'northwest', 'northeast']}

In [14]:
def binary_encode(df, column, positive_value):
    df = df.copy()
    df[column] = df[column].apply(lambda x: 1 if x == positive_value else 0)
    return df

def onehot_encode(df, column, prefix):
    df = df.copy()
    dummies = pd.get_dummies(df[column], dtype=int, prefix=prefix)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(column, axis=1)
    return df

In [19]:
def preprocess_inputs(df, scaler, train_size=0.7):

    df = df.copy()
    
    # Binary encode sex and smoker columns
    df = binary_encode(df, 'sex', 'Male')
    df = binary_encode(df, 'smoker', 'yes')

    # One-hot encode the region and children columns
    df = onehot_encode(df, 'region', 'R')
    df = onehot_encode(df, 'children', 'C')

    # Split df into X and y
    y = df['charges'].copy()
    X = df.drop('charges', axis=1).copy()

    # Scale X with the given scaler
    X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

    # Split data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_size, random_state=123)

    return X_train, X_test, y_train, y_test

In [16]:
data

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [29]:
X_train, X_test, y_train, y_test = preprocess_inputs(df=data, scaler=RobustScaler(), train_size=0.7)

In [30]:
X_train

,age,sex,bmi,smoker,R_northeast,R_northwest,R_southeast,R_southwest,C_0,C_1,C_2,C_3,C_4,C_5
300,-0.125000,0.0,-0.339387,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
904,0.875000,0.0,0.559690,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
670,-0.375000,0.0,0.139327,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
617,0.416667,0.0,-0.571599,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
373,-0.541667,0.0,0.297708,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1238,-0.083333,0.0,-0.916344,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1147,-0.791667,0.0,0.181006,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
106,-0.833333,0.0,-0.238166,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1041,-0.875000,0.0,-0.871093,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


### Training

In [31]:
models = {
    '         OLS Model': LinearRegression(),
    '          L2 Model': Ridge(),
    '          L1 Model': Lasso(),
    '  ElasticNet Model': ElasticNet(),
    '       L2 CV Model': RidgeCV(),
    '       L1 CV Model': LassoCV(),
    'ElasticNetCV Model': ElasticNetCV()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained.")

         OLS Model trained.
          L2 Model trained.
          L1 Model trained.
  ElasticNet Model trained.
       L2 CV Model trained.
       L1 CV Model trained.
ElasticNetCV Model trained.


In [32]:
print("Model R^2 Scores:\n------------------------------")
for name, model in models.items():
    print(name, model.score(X_test, y_test))

Model R^2 Scores:
------------------------------
         OLS Model 0.7593131310673423
          L2 Model 0.7593245099340612
          L1 Model 0.7593378044133104
  ElasticNet Model 0.34682689406085665
       L2 CV Model 0.7593245099340612
       L1 CV Model 0.759122785965993
ElasticNetCV Model 0.07190252821605014


In [ ]:
#### Standard Scaler scores

# Model R^2 Scores:
# ------------------------------
#          OLS Model 0.7593131310673424
#           L2 Model 0.7593171610896243
#           L1 Model 0.7593294653255952
#   ElasticNet Model 0.6721463640139935
#        L2 CV Model 0.759317161089625
#        L1 CV Model 0.7597041698503701
# ElasticNetCV Model 0.13927161105075647

In [ ]:
#### Min Max Scaler scores

# Model R^2 Scores:
# ------------------------------
#          OLS Model 0.7593131310673424
#           L2 Model 0.7595092777463504
#           L1 Model 0.7593526601115523
#   ElasticNet Model 0.3071833428091837
#        L2 CV Model 0.7595092777463504
#        L1 CV Model 0.759608350290162
# ElasticNetCV Model 0.05692460861159643